In [1]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

In [2]:
bank_user = pd.read_csv(r'bank-full.csv', sep=";")
df = bank_user.copy()
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [4]:
df.shape

(45211, 17)

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0


In [6]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

### The effect of variables on the dependent variable 'y' with the correlation matrix.

In [7]:
df['y'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

In [8]:
df_numeric = df.select_dtypes(include=('int64'))

In [9]:
df_numeric.head()

,age,balance,day,duration,campaign,pdays,previous,y
0,58,2143,5,261,1,-1,0,0
1,44,29,5,151,1,-1,0,0
2,33,2,5,76,1,-1,0,0
3,47,1506,5,92,1,-1,0,0
4,33,1,5,198,1,-1,0,0


In [10]:
correlation_matrix = df_numeric.corr()

# How strongly each numeric column moves with y
correlation_matrix['y'].drop('y').sort_values(ascending=False)

duration    0.394521
pdays       0.103621
previous    0.093236
balance     0.052838
age         0.025155
day        -0.028348
campaign   -0.073172
Name: y, dtype: float64

#### campaign and day are only weakly related to y, so we removed them.

In [11]:
df = df.drop(columns = ['campaign','day'])

# Examine each variable separately

## Age

In [12]:
df['age'].unique()

array([58, 44, 33, 47, 35, 28, 42, 43, 41, 29, 53, 57, 51, 45, 60, 56, 32,
       25, 40, 39, 52, 46, 36, 49, 59, 37, 50, 54, 55, 48, 24, 38, 31, 30,
       27, 34, 23, 26, 61, 22, 21, 20, 66, 62, 83, 75, 67, 70, 65, 68, 64,
       69, 72, 71, 19, 76, 85, 63, 90, 82, 73, 74, 78, 80, 94, 79, 77, 86,
       95, 81, 18, 89, 84, 87, 92, 93, 88], dtype=int64)

In [13]:
df['age'].value_counts()

age
32    2085
31    1996
33    1972
34    1930
35    1894
      ... 
93       2
90       2
95       2
88       2
94       1
Name: count, Length: 77, dtype: int64

## Job

In [14]:
df['job'].unique()

array(['management', 'technician', 'entrepreneur', 'blue-collar',
       'unknown', 'retired', 'admin.', 'services', 'self-employed',
       'unemployed', 'housemaid', 'student'], dtype=object)

In [15]:
df['job'].value_counts()

job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64

In [16]:
df = df[df['job'] != 'unknown']

## Marital

In [17]:
df['marital'].unique()

array(['married', 'single', 'divorced'], dtype=object)

## Education

In [18]:
df['education'].unique()

array(['tertiary', 'secondary', 'unknown', 'primary'], dtype=object)

In [19]:
df['education'].value_counts()

education
secondary    23131
tertiary     13262
primary       6800
unknown       1730
Name: count, dtype: int64

In [20]:
df = df[df['education'] != 'unknown']

## Default

In [21]:
df['default'].unique()

array(['no', 'yes'], dtype=object)

In [22]:
df['default'].value_counts()

default
no     42411
yes      782
Name: count, dtype: int64

## Housing	

In [23]:
df['housing'].unique()

array(['yes', 'no'], dtype=object)

In [24]:
df['housing'].value_counts()

housing
yes    24292
no     18901
Name: count, dtype: int64

## Loan

In [25]:
df['loan'].unique()

array(['no', 'yes'], dtype=object)

In [26]:
df['loan'].value_counts()

loan
no     36086
yes     7107
Name: count, dtype: int64

## Contact

In [27]:
df['contact'].unique()

array(['unknown', 'cellular', 'telephone'], dtype=object)

In [28]:
df['contact'].value_counts()

contact
cellular     28213
unknown      12286
telephone     2694
Name: count, dtype: int64

Updating the dataset with the new values after allocating the unknown category's values proportionately between telephone and cellular based on their current counts in the dataset.

In [29]:
contact_counts = df['contact'].value_counts()

unknown_count = contact_counts['unknown']
cellular_count = contact_counts['cellular']
telephone_count = contact_counts['telephone']

total = cellular_count + telephone_count
cellular_ratio = cellular_count / total
telephone_ratio = telephone_count / total

cellular_add = int(unknown_count * cellular_ratio)
telephone_add = int(unknown_count * telephone_ratio)

# Only the rows whose contact is unknown are changed
unknown_rows = df.index[df['contact'] == 'unknown']
df.loc[unknown_rows[:telephone_add], 'contact'] = 'telephone'
df.loc[unknown_rows[telephone_add:], 'contact'] = 'cellular'

new_contact_counts = df['contact'].value_counts()
print(new_contact_counts)

contact
cellular     39429
telephone     3764
Name: count, dtype: int64


## Month

In [30]:
df['month'].unique()

array(['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'jan', 'feb',
       'mar', 'apr', 'sep'], dtype=object)

In [31]:
df['month'].value_counts()

month
may    13192
jul     6601
aug     6037
jun     4980
nov     3842
apr     2820
feb     2533
jan     1318
oct      690
sep      532
mar      448
dec      200
Name: count, dtype: int64

## Duration

In [32]:
df['duration'].unique()

array([ 261,  151,   76, ..., 1298, 1246, 1556], dtype=int64)

In [33]:
df['duration'].value_counts()

duration
124     177
89      172
90      172
114     172
122     169
       ... 
1327      1
1331      1
1608      1
1833      1
1556      1
Name: count, Length: 1558, dtype: int64

## Pdays

In [34]:
df['pdays'].describe()

count    43193.000000
mean        40.404070
std        100.420624
min         -1.000000
25%         -1.000000
50%         -1.000000
75%         -1.000000
max        871.000000
Name: pdays, dtype: float64

## Previous

In [35]:
df['previous'].describe()

count    43193.000000
mean         0.584863
std          2.332672
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        275.000000
Name: previous, dtype: float64

## Poutcome

In [36]:
df['poutcome'].unique()

array(['unknown', 'failure', 'other', 'success'], dtype=object)

In [37]:
df['poutcome'].value_counts()

poutcome
unknown    35286
failure     4709
other       1774
success     1424
Name: count, dtype: int64

In [38]:
df['poutcome'] = df['poutcome'].replace('unknown', 'other')

print(df['poutcome'].value_counts())

poutcome
other      37060
failure     4709
success     1424
Name: count, dtype: int64


# y

In [39]:
df['y'].unique()

array([0, 1], dtype=int64)

In [40]:
df['y'].value_counts()

y
0    38172
1     5021
Name: count, dtype: int64

# Save the cleaned data

In [41]:
# Everything after this notebook starts from this file, not from bank-full.csv
os.makedirs('Final tables', exist_ok=True)
df.to_csv('Final tables/cleaned-data.csv', index=False)

print(f"Rows: {len(df)} (started with {len(bank_user)})")
print(f"Said yes: {df['y'].sum()} ({df['y'].mean():.1%})")
print(f"Columns: {list(df.columns)}")

Rows: 43193 (started with 45211)
Said yes: 5021 (11.6%)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'month', 'duration', 'pdays', 'previous', 'poutcome', 'y']


# Label Encoder

In [42]:
encoder = LabelEncoder()

label_mappings = {}

for col in df.select_dtypes('object'):
    df[col] = encoder.fit_transform(df[col])
    label_mappings[col] = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

for col, mapping in label_mappings.items():
    print(f"Column: {col}")
    for label, code in mapping.items():
        print(f"{code} -> {label}")
    print()

Column: job
0 -> admin.
1 -> blue-collar
2 -> entrepreneur
3 -> housemaid
4 -> management
5 -> retired
6 -> self-employed
7 -> services
8 -> student
9 -> technician
10 -> unemployed

Column: marital
0 -> divorced
1 -> married
2 -> single

Column: education
0 -> primary
1 -> secondary
2 -> tertiary

Column: default
0 -> no
1 -> yes

Column: housing
0 -> no
1 -> yes

Column: loan
0 -> no
1 -> yes

Column: contact
0 -> cellular
1 -> telephone

Column: month
0 -> apr
1 -> aug
2 -> dec
3 -> feb
4 -> jan
5 -> jul
6 -> jun
7 -> mar
8 -> may
9 -> nov
10 -> oct
11 -> sep

Column: poutcome
0 -> failure
1 -> other
2 -> success

